# ovarian-agent-contract: LLM extraction on a Colab GPU

Runs the pinned open-weight model (vLLM, JSON-schema constrained decoding, token logprobs) on the **dev and
calibration sites only**. Test sites stay locked in code until `eval/PROTOCOL.md` is frozen.

**Do not run this template.** Run the generated copy instead:
1. On your laptop: `uv run python -m oac.bundle`. It writes `dist/colab_extract_run.ipynb`, which has the code
   embedded, so nothing has to be uploaded.
2. In VS Code (Google Colab extension): open that file, then Select Kernel > Colab > New Colab Server > GPU.
   A T4 is enough; an L4 is about 2-3x faster. Keep one GPU type for a whole run (sections 7 and 8), because
   different GPUs use different kernels and can change outputs slightly. The manifest records the GPU.
3. Run the cells in order and save the notebook (Cmd+S) so the outputs are kept in the file.

**After a disconnect:** run every cell up to and including section 5 again (setup, a few minutes), skip the smoke test (section 6), then re-run the
section you were on. Finished reports are on Drive and are skipped. Keep the laptop awake during long runs.

Results go to Google Drive (`MyDrive/oac/runs/`). They contain report quotes, so keep them out of git (the repo
ignores `runs/`). Every cell is safe to re-run: the runner skips reports that are already done.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 1. Code (embedded by `oac.bundle`)

In [ ]:
import base64, io, os, shutil, zipfile
BUNDLE_B64 = "__BUNDLE_B64__"
CODE = "/content/oac"
if BUNDLE_B64.startswith("__"):
    raise SystemExit("This is the template. Run dist/colab_extract_run.ipynb made by `python -m oac.bundle`.")
if os.path.exists(CODE):
    shutil.rmtree(CODE)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))).extractall(CODE)
print("code commit:", open(f"{CODE}/COMMIT").read().strip())

## 2. Google Drive (where results are stored)

In the VS Code extension, `drive.mount()` may not work. If this cell fails, open the Command Palette and run
**"Colab: Mount Google Drive to Server..."**, then run this cell again.

In [ ]:
if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("drive.mount failed:", repr(e)[:200])
        print("Use the Command Palette: 'Colab: Mount Google Drive to Server...', then re-run this cell.")
if os.path.exists("/content/drive/MyDrive"):
    RUNS = "/content/drive/MyDrive/oac/runs"
else:
    RUNS = "/content/runs"
    print("WARNING: Drive is not mounted. Results stay on this server and are lost when it stops.")
os.makedirs(RUNS, exist_ok=True)
print("results ->", RUNS)

## 3. Install

vLLM brings its own PyTorch build; this takes a few minutes. Pinned to the version of the first successful run
(20 Nov 2026, Tesla T4: vllm 0.34.0, torch 2.14.0+cu130).

In [ ]:
VLLM_SPEC = "vllm==0.34.0"
!pip -q install "{VLLM_SPEC}"
!pip -q install -e {CODE}
!python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 4. Public data (sha256-checked)

In [ ]:
os.environ["OAC_DATA_DIR"] = "/content/data"
!cd {CODE} && python -m oac.fetch

## 5. Pinned model

Qwen3-8B-AWQ (Apache-2.0), fixed to one Hugging Face commit. It fits a 16 GB T4 in float16, so results do not
depend on which GPU Colab assigns. Thinking mode is off (`enable_thinking=False`).

In [ ]:
MODEL = "Qwen/Qwen3-8B-AWQ"
REVISION = "4da05a8edb55c6046cce958586c33b61da07bb79"
COMMON = f"--backend vllm --model {MODEL} --revision {REVISION} --dtype half --max-model-len 12288"

## 6. Smoke test on 3 synthetic reports (run it: checks token features and evidence bounds)

If this fails, save the notebook and tell Claude Code; it reads the error from this file. Common T4 issues:
out of memory (add `--gpu-mem 0.85` or `--max-model-len 6144` to COMMON) or an attention backend that does not
support the GPU.

In [ ]:
!cd {CODE} && rm -rf /content/smoke && python -m oac.run_llm {COMMON} --synthetic --k 2 --out /content/smoke 2>&1 | tail -40
import json
for line in open("/content/smoke/outputs.jsonl"):
    o = json.loads(line)
    print(o["case_id"], {f["field"]: (f["status"], f["value"]) for f in o["fields"]})
raws = [json.loads(l) for l in open("/content/smoke/raw.jsonl")]
print("token features computed:", sum(r["greedy"]["tokens_aligned"] for r in raws), "/ 3 (must be 3 / 3)")
def _obj(t):
    try:
        return json.loads(t)
    except Exception:
        return {}
quotes = [f.get("evidence") for r in raws for c in [r["greedy"]] + r["samples"] for f in _obj(c["text"]).values()
          if isinstance(f, dict) and isinstance(f.get("evidence"), list)]
print("evidence bounds held:", all(len(q) <= 3 and all(len(x) <= 200 for x in q) for q in quotes),
      "| truncated outputs:", sum(c["finish_reason"] == "length" for r in raws for c in [r["greedy"]] + r["samples"]),
      "(must be True | 0)")

## 7. System run: hybrid extraction, greedy + 5 samples (dev + calibration sites, 184 reports)

Measured on a T4 (20 Nov, prompt v0.1): about 35 to 45 minutes per 32 reports, so roughly 4 hours for all 184.
Results are written every 16 reports; after a disconnect, re-run setup and this cell and it continues.
Measured on an L4 (21 Nov, prompt v0.2): about 45 minutes for all 184. Each prompt version writes to its own
folders; earlier runs are kept.

In [ ]:
!cd {CODE} && python -m oac.run_llm {COMMON} --splits dev,calibration --mode hybrid --k 5 --batch 16 --out {RUNS}/qwen3-8b-awq_p0.3_hybrid_k5

## 8. Baseline 2: end-to-end (the model also stages directly), greedy only

In [ ]:
!cd {CODE} && python -m oac.run_llm {COMMON} --splits dev,calibration --mode e2e --k 0 --batch 16 --out {RUNS}/qwen3-8b-awq_p0.3_e2e

## 9. Summary (statuses only, no report text) and archive

Download `runs.zip` from Google Drive (`MyDrive/oac/`) and unzip it into the repo's `runs/` folder.

In [ ]:
import collections, glob
for p in sorted(glob.glob(f"{RUNS}/*/outputs.jsonl")):
    c = collections.Counter()
    n = 0
    for line in open(p):
        n += 1
        for f in json.loads(line)["fields"]:
            c[f["status"]] += 1
    print(p.split("/")[-2], n, "reports", dict(c))
for m in sorted(glob.glob(f"{RUNS}/*/manifest.json")):
    r = json.load(open(m))["runs"][-1]
    print(m.split("/")[-2], "| started", r["started_utc"], "| finished", r.get("finished_utc"), "|", r["backend"].get("gpu"), r["backend"].get("vllm"))
archive = shutil.make_archive(os.path.join(os.path.dirname(RUNS), "runs"), "zip", RUNS)
print("wrote", archive)